In [ ]:
#@title 個人の Google 連携を準備する（REVERSE） { display-mode: "form" }
# このノートは、REVERSE の設定画面の「Colab で自動準備」ボタンで開いたときだけ動きます（google-client-owned-onboarding-spec.md §5・§6.1）。
# - 戻り先と準備 ID は、開いた REVERSE の画面から受け取ります（入力は要りません）。戻り先は REVERSE の開発元が署名した証明で確かめます。
# - 呼ぶ先は下の許可リストだけです（Google ドライブは呼びません）。請求先は設定しません。
# - 作った設定（クライアント ID とシークレット）は、確かめた戻り先の REVERSE にだけ直接送ります。画面と保存されるノートには出しません。
import base64
import json
import re
import secrets
import time
import uuid
from urllib.parse import quote, urlsplit

TEMPLATE_VERSION = "2026-11-07.1"
# @generated-keys: apps/client/scripts/google-colab-setup/build-notebook.mjs が registry（purpose installation_origin）から埋める。
INSTALLATION_ORIGIN_KEYS = {}
REVOKED_KEY_IDS = []
# @end-generated-keys
HELLO = "reverse.google-colab-setup.hello/1"
PARAMS = "reverse.google-colab-setup.params/1"
SIGNING_DOMAIN = "baku.efo.sig/1"
PROOF_SCHEMA = "baku.efo.installation-origin/1"
CRM = "https://cloudresourcemanager.googleapis.com/v3/"
FIREBASE = "https://firebase.googleapis.com/"
IDENTITY = "https://identitytoolkit.googleapis.com/"
SERVICE_USAGE = "https://serviceusage.googleapis.com/v1/"
BILLING = "https://cloudbilling.googleapis.com/v1/"
USERINFO = "https://openidconnect.googleapis.com/v1/userinfo"
SETUP_APIS = ["firebase.googleapis.com", "identitytoolkit.googleapis.com"]
BUSINESS_APIS = [
    "drive.googleapis.com", "gmail.googleapis.com", "calendar-json.googleapis.com", "sheets.googleapis.com",
    "docs.googleapis.com", "slides.googleapis.com", "forms.googleapis.com", "people.googleapis.com",
    "tasks.googleapis.com", "meet.googleapis.com",
]
ALLOWED_REASONS = {
    "SERVICE_DISABLED", "ACCESS_TOKEN_SCOPE_INSUFFICIENT", "PERMISSION_DENIED", "TERMS_NOT_ACCEPTED", "TOS_NOT_ACCEPTED",
    "FIREBASE_TOS_NOT_ACCEPTED", "BILLING_DISABLED", "ORG_RESTRICTION_VIOLATION", "PROJECT_QUOTA_EXCEEDED", "CONSUMER_INVALID",
    "PROJECT_CREATION_DENIED", "QUOTA_EXCEEDED", "RESOURCE_EXHAUSTED", "INVALID_ARGUMENT", "ALREADY_EXISTS", "NOT_FOUND",
    "UNAUTHENTICATED", "FAILED_PRECONDITION", "INTERNAL", "UNAVAILABLE", "RATE_LIMIT_EXCEEDED", "USER_PROJECT_DENIED",
    "BILLING_NOT_ACTIVE",
}
GUIDANCE = {
    "TOS_NOT_ACCEPTED": "Google Cloud の利用規約への同意が必要です。https://console.cloud.google.com/ を開いて同意してから、もう一度実行してください。",
    "TERMS_NOT_ACCEPTED": "Google Cloud の利用規約への同意が必要です。https://console.cloud.google.com/ を開いて同意してから、もう一度実行してください。",
    "FIREBASE_TOS_NOT_ACCEPTED": "Firebase の利用規約への同意が必要です。https://console.firebase.google.com/ を開いて同意してから、もう一度実行してください。",
    "google_terms_required": "Google の利用規約への同意が必要です。Google Cloud と Firebase の画面を開いて同意してから、もう一度実行してください。",
    "PROJECT_QUOTA_EXCEEDED": "作れるプロジェクトの数の上限に達しています。REVERSE の画面の「個人の Google アカウントで準備」（これまでの手順）を使ってください。",
    "ORG_RESTRICTION_VIOLATION": "このアカウントの組織の設定で、プロジェクトを作れません。個人の Google アカウントで実行してください。",
    "billing_linked_or_unknown": "このプロジェクトに請求先が設定されているか、確かめられませんでした。費用が出ないよう、ここで止めました。",
    "personal_project_required": "組織のフォルダーの下のプロジェクトになっています。個人の Google アカウントで実行してください。",
    "project_label_mismatch": "同じ名前のプロジェクトが、この準備で作ったものではありませんでした。REVERSE の画面で「最初からやり直す」を押してください。",
}


class SetupStop(Exception):
    def __init__(self, stage, reason, status=None):
        super().__init__(reason)
        self.stage = stage
        self.reason = reason
        self.status = status


def canonical(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"), ensure_ascii=False)


def https_origin(value):
    if not isinstance(value, str) or len(value) > 512 or not re.fullmatch(r"[\x21-\x7e]+", value):
        return None
    parts = urlsplit(value)
    if parts.scheme != "https" or not parts.hostname or parts.port or parts.username or parts.password:
        return None
    if parts.path not in ("", "/") or parts.query or parts.fragment:
        return None
    return "https://" + parts.hostname


def receive_params():
    """開いた REVERSE の画面へ秘密を含まない呼びかけを送り、戻り先と準備 ID と証明を受け取る。"""
    from google.colab import output
    script = """
    new Promise((resolve) => {
      let opener = null;
      try { opener = window.top.opener; } catch (e) { opener = null; }
      if (!opener) { resolve({ error: "no_opener" }); return; }
      const timer = setTimeout(() => resolve({ error: "timeout" }), 60000);
      window.addEventListener("message", (event) => {
        if (event.source !== opener) return;
        const data = event.data;
        if (!data || data.type !== "%s") return;
        clearTimeout(timer);
        resolve({ origin: event.origin, data: data });
      });
      opener.postMessage({ type: "%s" }, "*");
    })
    """ % (PARAMS, HELLO)
    result = output.eval_js(script, timeout_sec=90)
    if not isinstance(result, dict) or result.get("error"):
        raise SetupStop("receive_params", (result or {}).get("error", "no_response") if isinstance(result, dict) else "no_response")
    data = result.get("data")
    if not isinstance(data, dict):
        raise SetupStop("receive_params", "params_rejected")
    origin = https_origin(data.get("returnOrigin"))
    if not origin or origin != result.get("origin"):
        raise SetupStop("receive_params", "origin_mismatch")
    if data.get("schemaVersion") != 1 or data.get("templateVersion") != TEMPLATE_VERSION:
        raise SetupStop("receive_params", "template_version_mismatch")
    if not isinstance(data.get("preparationId"), str) or not re.fullmatch(r"[a-f0-9]{32}", data["preparationId"]):
        raise SetupStop("receive_params", "params_rejected")
    if type(data.get("generation")) is not int or not 1 <= data["generation"] <= 1000000:
        raise SetupStop("receive_params", "params_rejected")
    if not isinstance(data.get("projectId"), str) or not re.fullmatch(r"[a-z][a-z0-9-]{4,28}[a-z0-9]", data["projectId"]):
        raise SetupStop("receive_params", "params_rejected")
    if not isinstance(data.get("projectLabel"), str) or not re.fullmatch(r"[a-z0-9_-]{8,63}", data["projectLabel"]):
        raise SetupStop("receive_params", "params_rejected")
    app_name = data.get("appName")
    app_name = app_name.strip()[:40] if isinstance(app_name, str) and re.fullmatch(r"[^\x00-\x1f\x7f]{1,80}", app_name) else "REVERSE"
    verify_origin_proof(data.get("proof"), origin, int(time.time()))
    return {
        "origin": origin, "preparationId": data["preparationId"], "generation": data["generation"],
        "projectId": data["projectId"], "projectLabel": data["projectLabel"], "appName": app_name or "REVERSE",
    }


def verify_origin_proof(proof, origin, now):
    """戻り先が REVERSE の確認済みの設置であることを、開発元の公開鍵で確かめる（§6.1.1）。"""
    try:
        from cryptography.exceptions import InvalidSignature
        from cryptography.hazmat.primitives.asymmetric.ed25519 import Ed25519PublicKey
    except Exception:
        raise SetupStop("verify_origin", "verifier_unavailable")
    if not INSTALLATION_ORIGIN_KEYS:
        raise SetupStop("verify_origin", "origin_keys_not_published")
    if not isinstance(proof, dict) or set(proof) != {"evidence", "purpose", "signature", "keyId"}:
        raise SetupStop("verify_origin", "proof_rejected")
    evidence = proof["evidence"]
    if not isinstance(evidence, dict) or set(evidence) != {"schema", "keyId", "purpose", "licenseId", "origin", "iat", "exp"}:
        raise SetupStop("verify_origin", "proof_rejected")
    key_id = proof["keyId"]
    if (proof["purpose"] != "installation_origin" or evidence["purpose"] != "installation_origin" or evidence["schema"] != PROOF_SCHEMA
            or evidence["keyId"] != key_id or key_id in REVOKED_KEY_IDS or key_id not in INSTALLATION_ORIGIN_KEYS):
        raise SetupStop("verify_origin", "proof_rejected")
    if evidence["origin"] != origin or type(evidence["iat"]) is not int or type(evidence["exp"]) is not int:
        raise SetupStop("verify_origin", "proof_rejected")
    if evidence["exp"] <= now or evidence["iat"] > now + 300:
        raise SetupStop("verify_origin", "proof_expired")
    try:
        public = Ed25519PublicKey.from_public_bytes(base64.urlsafe_b64decode(INSTALLATION_ORIGIN_KEYS[key_id] + "=="))
        signature = base64.b64decode(proof["signature"], validate=True)
        public.verify(signature, canonical({"d": SIGNING_DOMAIN, "p": "installation_origin", "m": evidence}).encode("utf-8"))
    except (InvalidSignature, ValueError, TypeError):
        raise SetupStop("verify_origin", "proof_signature_invalid")


def classified_error(data, stage, status):
    error = data.get("error", {}) if isinstance(data, dict) else {}
    error = error if isinstance(error, dict) else {}
    details = error.get("details", [])
    details = details if isinstance(details, list) else []
    reasons = [item.get("reason") for item in details if isinstance(item, dict)]
    reason = next((value for value in reasons if isinstance(value, str) and value in ALLOWED_REASONS), None)
    message = str(error.get("message", "")).lower()
    if reason is None and ("terms" in message or "tos" in message):
        reason = "google_terms_required"
    if reason is None and "billing" in message:
        reason = "billing_requirement_reported"
    if reason is None and error.get("status") in ALLOWED_REASONS:
        reason = error["status"]
    return SetupStop(stage, reason or "google_request_rejected", status)


def run_setup():
    params = receive_params()
    print("REVERSE の画面を確かめました（戻り先: " + params["origin"] + "）。", flush=True)
    import google.auth
    from google.auth.transport.requests import AuthorizedSession
    from google.colab import auth
    auth.authenticate_user()
    credentials, _ = google.auth.default()
    session = AuthorizedSession(credentials)
    project_id, label = params["projectId"], params["projectLabel"]
    callback = params["origin"] + "/api/connections/google/callback"
    state = {"number": None}

    def request(stage, url, method="GET", body=None):
        allowed = url.startswith((CRM, FIREBASE, IDENTITY, SERVICE_USAGE, BILLING)) or url == USERINFO
        if not allowed:
            raise SetupStop(stage, "request_destination_rejected")
        headers = {"x-goog-user-project": state["number"]} if state["number"] and url.startswith((FIREBASE, IDENTITY)) else {}
        response = session.request(method, url, json=body, headers=headers, timeout=20, allow_redirects=False)
        if 300 <= response.status_code < 400:
            raise SetupStop(stage, "redirect_rejected", response.status_code)
        if len(response.content) > 65536:
            raise SetupStop(stage, "response_size_rejected", response.status_code)
        try:
            data = response.json()
        except Exception:
            raise SetupStop(stage, "response_format_rejected", response.status_code)
        if not isinstance(data, dict):
            raise SetupStop(stage, "response_format_rejected", response.status_code)
        if not 200 <= response.status_code < 300:
            raise classified_error(data, stage, response.status_code)
        return data

    def operation_name(data, stage):
        name = data.get("name") if isinstance(data, dict) else None
        if (not isinstance(name, str) or len(name) > 2048
                or not re.fullmatch(r"(?:operations|projects/[A-Za-z0-9_-]+/operations)/[A-Za-z0-9._~+=:@/-]+", name)
                or any(part in {".", ".."} for part in name.split("/"))):
            raise SetupStop(stage, "operation_identifier_rejected")
        return name

    def poll(base, name, stage):
        checked = operation_name({"name": name}, stage)
        path = "/".join(quote(part, safe="") for part in checked.split("/"))
        deadline = time.monotonic() + 180
        while time.monotonic() < deadline:
            data = request(stage, base + path)
            if data.get("name") != checked or ("done" in data and type(data["done"]) is not bool):
                raise SetupStop(stage, "operation_response_rejected")
            if data.get("done") is True:
                if "error" in data:
                    raise classified_error({"error": data["error"]}, stage, 200)
                return
            time.sleep(3)
        raise SetupStop(stage, "operation_still_running")

    def billing_unlinked():
        info = request("verify_billing", BILLING + "projects/" + project_id + "/billingInfo")
        if (info.get("name") != "projects/" + project_id + "/billingInfo" or info.get("billingEnabled") not in (False, None)
                or info.get("billingAccountName") not in (None, "")):
            raise SetupStop("verify_billing", "billing_linked_or_unknown")

    try:
        identity = request("verify_identity", USERINFO)
        email = identity.get("email") if identity.get("email_verified") is True else None
        email = email if isinstance(email, str) and re.fullmatch(r"[A-Za-z0-9._%+-]{1,64}@[A-Za-z0-9.-]{1,253}\.[A-Za-z]{2,24}", email) else None
        print("1/5 専用のプロジェクトを用意しています…", flush=True)
        try:
            created = request("create_project", CRM + "projects", "POST", {
                "projectId": project_id, "displayName": "REVERSE Google Connection", "labels": {"reverse-setup": label},
            })
            poll(CRM, operation_name(created, "create_project"), "poll_project")
        except SetupStop as stop:
            if stop.reason != "ALREADY_EXISTS" and stop.status != 409:
                raise
        project = None
        for _ in range(20):
            project = request("verify_project", CRM + "projects/" + project_id)
            if project.get("state") == "ACTIVE":
                break
            time.sleep(3)
        number = str(project.get("name", "")).removeprefix("projects/")
        if project.get("projectId") != project_id or project.get("state") != "ACTIVE" or not re.fullmatch(r"[1-9][0-9]{0,19}", number):
            raise SetupStop("verify_project", "project_not_ready")
        if (project.get("labels") or {}).get("reverse-setup") != label:
            raise SetupStop("verify_project", "project_label_mismatch")
        if project.get("parent"):
            raise SetupStop("verify_project", "personal_project_required")
        state["number"] = number
        billing_unlinked()
        print("2/5 必要な API を有効にしています…", flush=True)
        services = SETUP_APIS + BUSINESS_APIS

        def service_states():
            found = {}
            for service in services:
                data = request("verify_service", SERVICE_USAGE + "projects/" + number + "/services/" + service + "?fields=name,state")
                if data.get("name") != "projects/" + number + "/services/" + service or data.get("state") not in {"ENABLED", "DISABLED"}:
                    raise SetupStop("verify_service", "service_state_unreadable")
                found[service] = data["state"]
            return found

        pending = [service for service, value in service_states().items() if value != "ENABLED"]
        if pending:
            enabled = request("enable_apis", SERVICE_USAGE + "projects/" + number + "/services:batchEnable", "POST", {"serviceIds": pending})
            try:
                poll(SERVICE_USAGE, operation_name(enabled, "enable_apis"), "poll_api_enablement")
            except SetupStop as stop:
                if stop.status != 403:
                    raise
        for _ in range(20):
            if all(value == "ENABLED" for value in service_states().values()):
                break
            time.sleep(3)
        else:
            raise SetupStop("verify_service", "apis_not_all_enabled")
        billing_unlinked()
        print("3/5 自分専用の連携アプリを作っています…", flush=True)
        provisioned = request("provision_firebase", FIREBASE + "v1alpha/firebase:provisionFirebaseApp", "POST", {
            "parent": "projects/" + project_id,
            "appNamespace": "reverse-google-setup-" + label[-12:],
            "displayName": "REVERSE",
            "requestId": str(uuid.uuid4()),
            "webInput": {},
            "firebaseAuthInput": {
                "googleSigninProviderMode": "PROVIDER_ENABLED",
                "googleSigninProviderConfig": {
                    "publicDisplayName": params["appName"],
                    **({"customerSupportEmail": email} if email else {}),
                    "oauthRedirectUris": [callback],
                },
            },
        })
        try:
            poll(FIREBASE + "v1beta1/", operation_name(provisioned, "provision_firebase"), "poll_firebase")
        except SetupStop as stop:
            if stop.reason != "USER_PROJECT_DENIED" or stop.status != 403:
                raise
        firebase_project = request("verify_firebase", FIREBASE + "v1beta1/projects/" + project_id)
        if firebase_project.get("projectId") != project_id or str(firebase_project.get("projectNumber")) != number or firebase_project.get("state") != "ACTIVE":
            raise SetupStop("verify_firebase", "firebase_project_mismatch")
        location = (firebase_project.get("resources") or {}).get("locationId")
        provider = request("read_google_provider", IDENTITY + "admin/v2/projects/" + project_id + "/defaultSupportedIdpConfigs/google.com")
        client_id, client_secret = provider.get("clientId"), provider.get("clientSecret")
        if (provider.get("name") not in {"projects/" + project_id + "/defaultSupportedIdpConfigs/google.com", "projects/" + number + "/defaultSupportedIdpConfigs/google.com"}
                or provider.get("enabled") is not True or not isinstance(client_id, str)
                or not re.fullmatch(re.escape(number) + r"-[A-Za-z0-9_-]{8,256}\.apps\.googleusercontent\.com", client_id)
                or not isinstance(client_secret, str) or not re.fullmatch(r"[A-Za-z0-9_-]{20,256}", client_secret)):
            raise SetupStop("read_google_provider", "google_provider_mismatch")
        billing_unlinked()
        print("4/5 REVERSE へ設定を送っています…", flush=True)
        pairing = "%06d" % secrets.randbelow(1000000)
        import requests
        response = requests.post(params["origin"] + "/api/connections/google/setup-handoff", json={
            "schemaVersion": 1, "preparationId": params["preparationId"], "generation": params["generation"],
            "templateVersion": TEMPLATE_VERSION, "pairingCode": pairing, "projectId": project_id, "projectNumber": number,
            "observedEmail": email, "clientId": client_id, "clientSecret": client_secret,
        }, timeout=20, allow_redirects=False)
        client_secret = None
        provider = None
        if response.status_code != 200 or len(response.content) > 4096:
            raise SetupStop("handoff", "reverse_did_not_accept", response.status_code)
        try:
            accepted = response.json().get("status") in {"accepted", "already_received"}
        except Exception:
            accepted = False
        if not accepted:
            raise SetupStop("handoff", "reverse_did_not_accept", response.status_code)
        return {"pairing": pairing, "projectId": project_id, "resourceLocationId": location}
    finally:
        session.close()


def show_pairing(result):
    from IPython.display import HTML, display
    display(HTML(
        "<div style='font-family:sans-serif;padding:16px;border:2px solid #1a73e8;border-radius:12px;max-width:560px'>"
        "<p style='margin:0 0 8px'>5/5 準備が終わりました。REVERSE の画面に戻り、同じ照合コードが出ているか確かめて「登録する」を押してください。</p>"
        "<p style='font-size:40px;letter-spacing:8px;margin:8px 0;font-weight:bold'>" + result["pairing"] + "</p>"
        "<p style='margin:0;color:#555'>プロジェクト: " + result["projectId"] + "（請求先は設定していません）</p></div>"
    ))


try:
    show_pairing(run_setup())
except SetupStop as stop:
    print("止まりました（" + stop.stage + ": " + stop.reason + "）。", flush=True)
    print(GUIDANCE.get(stop.reason, "REVERSE の画面に戻り、表示に沿ってやり直してください。何度も止まるときは、これまでの手順（「個人の Google アカウントで準備」）を使えます。"), flush=True)
except Exception:
    print("止まりました（理由を表示できない失敗です）。REVERSE の画面に戻り、もう一度お試しください。", flush=True)
